# AfriMeet AI — Whisper Fine-Tuning (Colab GPU)

Hybrid workflow: data download/prep can run locally or here; this notebook does the
GPU-heavy parts — Phase 3 (baseline evaluation), Phase 4 (fine-tuning), and Phase 5
(comparison) — using Colab's free GPU. It doesn't duplicate any pipeline logic: every
cell below just calls into the same `afrimeet` package and `scripts/` used locally.
Google Drive is used to persist data and checkpoints between sessions, since Colab's
local disk is wiped when the runtime disconnects.

**Before you start:**
1. `Runtime -> Change runtime type -> GPU` (T4 is fine).
2. The code is cloned from GitHub (`REPO_URL` below, defaults to this project's private
   repo). Since the repo is **private**, the clone cell will prompt you for a GitHub
   Personal Access Token (classic PAT with `repo` scope, or a fine-grained PAT scoped to
   just this repo with Contents: Read). Generate one at
   `github.com/settings/tokens` — the notebook only holds it in memory for the clone
   step via `getpass`, it's never written to disk or saved in the notebook.
   - If you'd rather not use a token, clear `REPO_URL` to fall back to the Drive-zip
     method: run `python scripts/package_for_colab.py` locally and upload
     `dist/afrimeet-ai-code.zip` to `My Drive/AfriMeet_AI/afrimeet-ai-code.zip`.

Phase 6 (the API / web app) isn't part of this notebook — it's meant to run wherever
you deploy it, using the fine-tuned model this notebook produces.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_NAME = "afrimeet-ai"
WORK_DIR = f"/content/{PROJECT_NAME}"
DRIVE_ROOT = "/content/drive/MyDrive/AfriMeet_AI"

# Private repo -> the clone cell below will prompt for a GitHub token via getpass.
# Clear this (set to "") to fall back to the Drive-zip method instead.
REPO_URL = "https://github.com/claverfred/afrimeet-ai.git"
CODE_ZIP_PATH = f"{DRIVE_ROOT}/afrimeet-ai-code.zip"

os.makedirs(DRIVE_ROOT, exist_ok=True)
print("WORK_DIR:", WORK_DIR)
print("DRIVE_ROOT:", DRIVE_ROOT)

In [ ]:
import getpass
import shutil
import subprocess

if REPO_URL:
    clone_url = REPO_URL
    token = ""
    if "github.com" in REPO_URL:
        token = getpass.getpass(
            "GitHub token (repo scope) for private clone — leave blank if the repo is public: "
        )
        if token:
            clone_url = REPO_URL.replace("https://", f"https://{token}@")
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    subprocess.run(["git", "clone", clone_url, WORK_DIR], check=True)
    del clone_url, token  # don't let the token linger in a variable after this cell
else:
    assert os.path.exists(CODE_ZIP_PATH), (
        f"{CODE_ZIP_PATH} not found. Run `python scripts/package_for_colab.py` "
        "locally and upload the resulting zip to that Drive path, or set REPO_URL above."
    )
    os.makedirs(WORK_DIR, exist_ok=True)
    shutil.unpack_archive(CODE_ZIP_PATH, WORK_DIR)
    print(f"Unpacked {CODE_ZIP_PATH} -> {WORK_DIR}")

In [ ]:
%cd $WORK_DIR
!pip install -q -r requirements/ml.txt -r requirements/api.txt
!pip install -q -e . --no-deps

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

import afrimeet
print("afrimeet package loaded, version", afrimeet.__version__)

## Phase 2 — Data

Restores the processed dataset from a Drive-cached zip if one exists (fast, no
re-download). Otherwise downloads + prepares it fresh on the local Colab disk (fast
I/O — writing thousands of small audio files directly to a Drive-mounted path is
much slower) and caches the result back to Drive for next time.

In [ ]:
import shutil
from pathlib import Path

data_zip = Path(DRIVE_ROOT) / "data_processed.zip"
processed_dir = Path(WORK_DIR) / "data" / "processed"

if data_zip.exists():
    print(f"Restoring processed dataset from {data_zip} ...")
    processed_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(data_zip), str(processed_dir))
else:
    print("No cached dataset on Drive — downloading and preparing from scratch.")
    !python scripts/download_data.py
    !python scripts/prepare_dataset.py
    print(f"Caching processed dataset to {data_zip} for future sessions ...")
    shutil.make_archive(str(data_zip.with_suffix("")), "zip", root_dir=str(processed_dir))

## Phase 3 — Baseline evaluation (pre-trained Whisper)

In [ ]:
import glob

test_manifests = glob.glob(f"{WORK_DIR}/data/processed/*/test/manifest.csv")
assert test_manifests, "No test manifest found under data/processed/*/test/manifest.csv"
TEST_MANIFEST = test_manifests[0]
print("Using test manifest:", TEST_MANIFEST)

In [ ]:
!python scripts/evaluate.py --manifest "$TEST_MANIFEST" --model openai/whisper-small --run-name baseline

## Phase 4 — Fine-tune Whisper on the conference-domain data

Hyperparameters come from `configs/config.yaml` (`training:` section). Lower
`train_batch_size` there if you hit a CUDA out-of-memory error on the T4's 16GB.

If a fine-tuned model backup already exists on Drive from a previous session, this
restores it instead of retraining — set `RETRAIN = True` to force a fresh run.

In [ ]:
import shutil
from pathlib import Path

finetuned_dir = Path(WORK_DIR) / "models" / "finetuned"
backup_zip = Path(DRIVE_ROOT) / "models_finetuned.zip"

RETRAIN = False

if backup_zip.exists() and not RETRAIN:
    print(f"Found existing backup at {backup_zip} — restoring instead of retraining.")
    finetuned_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(backup_zip), str(finetuned_dir))
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False

In [ ]:
if not SKIP_TRAINING:
    !python scripts/train.py
else:
    print("Skipping training — using the model restored from Drive. Set RETRAIN = True above to force retraining.")

In [ ]:
# Back up the fine-tuned model to Drive (safe to re-run any time, including mid-training
# from a second cell if you're worried about a disconnect on a long run).
backup_path = shutil.make_archive(str(backup_zip.with_suffix("")), "zip", root_dir=str(finetuned_dir))
print(f"Backed up fine-tuned model to {backup_path}")

## Phase 5 — Evaluate the fine-tuned model and compare against baseline

In [ ]:
from afrimeet.utils.config import load_config

config = load_config()
FINETUNED_MODEL = f"{config['paths']['models_finetuned']}/{config['whisper']['finetuned_model_name']}"
print("Using fine-tuned model:", FINETUNED_MODEL)

In [ ]:
!python scripts/evaluate.py --manifest "$TEST_MANIFEST" --model "$FINETUNED_MODEL" --run-name finetuned

In [ ]:
!python scripts/compare_models.py --runs baseline=reports/metrics/baseline_summary.json finetuned=reports/metrics/finetuned_summary.json

In [ ]:
import shutil
from pathlib import Path

reports_src = Path(WORK_DIR) / "reports" / "metrics"
reports_backup = Path(DRIVE_ROOT) / "reports_metrics"
if reports_backup.exists():
    shutil.rmtree(reports_backup)
shutil.copytree(reports_src, reports_backup)
print(f"Backed up metrics to {reports_backup}")

## Resuming in a later session

Just re-run the cells from the top:
- The data cell finds `data_processed.zip` on Drive and skips re-downloading.
- The training cell finds `models_finetuned.zip` on Drive and skips retraining
  (restores the model instead). Set `RETRAIN = True` to fine-tune again — e.g. after
  changing hyperparameters in `configs/config.yaml`.

For a long training run, Trainer also checkpoints locally to
`models/finetuned/.../checkpoint-N` every `save_steps` — but that's on the ephemeral
Colab disk, so it's lost on a disconnect mid-run. Re-run the "back up the fine-tuned
model to Drive" cell periodically during a long run if you want that protection.